# 01 · Veri hazırlama (Makale 1)

**Akıştaki yeri:** `01 veri` → 02 görsel model → 03 VLM/LLM → 04 RAG → 05 değerlendirme

Ne yapar: BRIGHT'ın Türkiye 2023 deprem hasar etiketlerini (DOI 10.5194/essd-17-6217-2025; veri DOI 10.5281/zenodo.14619797) alır, her karo için Maxar Open Data'dan olay-sonrası **optik** görüntüyü keser ve `Veri/yedek_bright_maxar_turkiye/` altına yazar.

Çıktı:
- `pre/` öncesi optik, `post/` sonrası optik (JPEG), `label/` **ham sınıf indeksi** (PNG; 0 arka plan, 1 sağlam, 2 hasarlı, 3 yıkık), `manifest.csv`.
- `label_renkli/` ve `overlay/` yalnızca **gözle kontrol** içindir (2b'ye bakın); model girdisi/hedefi değildir.
- `post_hizali/` ve `overlay_hizali/`: sonrası görüntünün, şehir başına ölçülen kaymayla **etiketin/öncesinin geometrisine taşınmış** kopyası (5c; xBD ve BRIGHT'taki gibi referans öncesi/etiket). `label/` ve `pre/` her zaman değişmeden kalır.

Notlar:
- Yeniden başlatılabilir: `manifest.csv`'de olan karolar atlanır; yarıda kesilirse yeniden çalıştırın.
- Lisans: Maxar Open Data CC BY-NC 4.0; BRIGHT etiketleri görüntü lisansını izler. Yayından önce kontrol edin.
- Öncesi görüntülerin bir kısmı kaynakta (BRIGHT) kısmen boştur (nodata=0, siyah); `manifest.csv` içindeki `pre_blank_frac` bunu ölçer (5. ve 6. bölüm).
- Sonrası görüntü bakış açısı şehre göre 6°–44° arasında değişir; bu bölmeyi ve bina düzeyi kıyası etkiler (6. bölüme bakın).

In [ ]:
%pip install -q rasterio remotezip tifffile pillow numpy imagecodecs pandas matplotlib

## 1. Ayarlar

In [ ]:
import csv, io, json, math, os, time, threading, urllib.request, zipfile, hashlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urljoin
import numpy as np, rasterio, tifffile
from PIL import Image
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from remotezip import RemoteZip

# --- yollar ---
# Bu not defteri Kod/M1/ içinde durur, proje kökü iki üsttedir. (Eski sürüm Path.cwd().parent kullanıyordu; bu Kod/ klasörünü
# verdiği için çıktı yanlışlıkla Kod/Veri/ altına yazılıyordu.) "Kaç üst" saymak kırılgan olduğundan bulunduğumuz klasörden yukarı
# çıkarak Kod/ ve Veri/ klasörlerini birlikte içeren ilk klasörü proje kökü sayarız; not defteri nereden açılırsa açılsın çalışır.
def proje_koku():
    if os.environ.get("RAG_PROJE"):
        return Path(os.environ["RAG_PROJE"]).resolve()
    bu = Path.cwd().resolve()
    for d in [bu, *bu.parents]:
        if (d / "Kod").is_dir() and (d / "Veri").is_dir():
            return d
    raise RuntimeError("Proje kökü bulunamadı (Kod/ ve Veri/ içeren klasör). RAG_PROJE ortam değişkenini ayarlayın.")

PROJE = proje_koku()
OUT = Path(os.environ.get("RAG_OUT", PROJE / "Veri" / "yedek_bright_maxar_turkiye"))
SCRATCH = Path(os.environ.get("RAG_SCRATCH", Path.home() / "scratch_bright"))   # geçici dosyalar proje dışında

# --- çalıştırma ayarları ---
LIMIT = int(os.environ.get("RAG_LIMIT", 0))   # 0 = hepsi; denemek için örn. 5
MAX_SECONDS = None                              # uzun çağrı sınırı olan ortamlarda örn. 120
WORKERS = 4                                     # 10'a çıkarmak Maxar/GDAL tarafında takılmalara yol açtı

# --- kaynaklar ---
MAXAR = "https://maxar-opendata.s3.amazonaws.com/events/Kahramanmaras-turkey-earthquake-23/"
HF = "https://huggingface.co/datasets/Kullervo/BRIGHT/resolve/main/"
POST_FROM, POST_PREFERRED_UNTIL = "2023-02-06", "2023-02-15"   # tercihen ilk 9 gün, en küçük off-nadir

# --- şehir düzeyinde bölme (en yakın şehir merkezi; boylam, enlem) ---
CITIES = {'Antakya/Defne': (36.16, 36.20), 'Iskenderun': (36.17, 36.59), 'Kirikhan': (36.36, 36.50),
          'Samandag': (35.97, 36.08), 'Hassa': (36.52, 36.80), 'Kahramanmaras': (36.93, 37.58),
          'Elbistan': (37.19, 38.20), 'Pazarcik': (37.29, 37.48), 'Nurdagi': (36.73, 37.17),
          'Islahiye': (36.63, 37.02), 'Gaziantep': (37.38, 37.06), 'Osmaniye': (36.25, 37.07),
          'Adiyaman': (38.28, 37.76), 'Malatya': (38.31, 38.35), 'Turkoglu': (36.85, 37.38),
          'Diyarbakir': (40.22, 37.91), 'Sanliurfa': (38.80, 37.16), 'Kilis': (37.12, 36.72), 'Adana': (35.32, 37.0)}
SPLIT = {'Adiyaman': 'dev', 'Malatya': 'dev', 'Kahramanmaras': 'dev', 'Islahiye': 'dev',
         'Antakya/Defne': 'test', 'Osmaniye': 'test', 'Gaziantep': 'test'}

for d in ("pre", "post", "label", "label_renkli", "overlay"):
    (OUT / d).mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True); (SCRATCH / "stac_cache").mkdir(exist_ok=True)
print("Proje kökü:", PROJE)
print("Çıktı:", OUT)

## 2. Yardımcılar (önbellekli HTTP ve yeniden deneme)

In [ ]:
def retry(f, n=5):
    for t in range(n):
        try:
            return f()
        except Exception as e:
            err = e; time.sleep(3 + 4 * t)
    raise err

def http_json(u):
    p = SCRATCH / "stac_cache" / (hashlib.md5(u.encode()).hexdigest() + ".json")
    if p.exists():
        return json.loads(p.read_text(encoding="utf8"))
    d = retry(lambda: json.load(urllib.request.urlopen(u, timeout=60)))
    p.write_text(json.dumps(d), encoding="utf8")
    return d

def dist_km(a, b):
    return math.hypot((a[0] - b[0]) * math.cos(math.radians(37)), a[1] - b[1]) * 111

## 2b. Etiket görselleştirme ve manifest yardımcıları

**Etiketler neden siyah görünür?** `label/*.png` her piksel için sınıf **indeksini** (0, 1, 2, 3) saklayan tek kanallı bir PNG'dir. Görüntü izleyiciler 8 bitlik gri değeri 0–255 ölçeğinde çizer; 0–3 değerleri bu ölçekte (en parlak değerin yüzde birinden azı) neredeyse siyahtır. Dosya bozuk değildir: içinde 1, 2, 3 değerleri vardır, yalnızca gözle okunamaz.

Bu yüzden iki ayrı çıktı tutuyoruz ve aralarındaki sınırı net tutuyoruz:

| Klasör | Kimin için | İçerik |
|---|---|---|
| `label/` | makine (eğitim, değerlendirme) | ham indeks 0–3, **tek gerçek etiket** |
| `label_renkli/` | insan | aynı etiket, palet ile renkli (0 siyah, 1 yeşil, 2 sarı, 3 kırmızı) |
| `overlay/` | insan | etiketli pikseller sonrası görüntünün üstüne yarı saydam boyanmış |

Kural: model ve metrik kodu **yalnızca `label/`** okur. Renkli kopyalar JPEG/renk dönüşümü içerdiği için geri indekse çevrilmemelidir.

Renklendirme `PALET[L]` ile yapılır: `PALET` 4 satırlı bir renk tablosudur, `L` indeks dizisidir; numpy her pikseli kendi indeksinin satırıyla değiştirir (döngü yok, tek satır, hızlı).

**Manifest yükseltme:** Eski manifest `pre_blank_frac` sütununu içermiyordu ve hatalı/kapsamsız satırları da "bitmiş" sayıyordu (bu yüzden bir daha denenmiyorlardı). `manifest_oku_yukselt` (1) yalnız `status=ok` satırlarını tutar, (2) eksik `pre_blank_frac` değerini kayıtlı `pre/*.jpg`'den hesaplar, (3) dosyayı güncel şemayla yeniden yazar.

**Öncesi görüntüdeki siyah alan:** BRIGHT'ın öncesi görüntüleri kaynakta `nodata=0` ile gelir ve bazı karolarda alanın büyük kısmı boştur. Bu bizim kesmemizden değil kaynaktan gelir. `bos_oran`, RGB toplamı 15'in altındaki piksellerin oranını verir (JPEG sıkıştırması saf siyahın kenarlarını biraz bulandırdığı için tam sıfır yerine küçük bir eşik). Yeni üretimde değer ham GeoTIFF'ten, eski kayıtlarda JPEG'den hesaplanır; ikisi aynı ölçütü kullanır, fark binde birler düzeyindedir.

In [ ]:
# Sınıf paleti: indeks -> RGB. Yalnızca gözle bakmak için; gerçek etiket label/ içindeki ham indekstir.
SINIF_ADLARI = {0: "arka plan", 1: "sağlam", 2: "hasarlı", 3: "yıkık"}
PALET = np.array([[0, 0, 0], [0, 200, 0], [255, 200, 0], [255, 0, 0]], dtype=np.uint8)

def etiket_renklendir(L):
    """(H, W) uint8 sınıf indeksi (0-3) -> (H, W, 3) uint8 RGB."""
    if L.max() > 3:
        raise ValueError(f"beklenmeyen sınıf indeksi: {np.unique(L)}")
    return PALET[L]

def bindir(post, L, alfa=0.45):
    """Etiketli (L > 0) pikselleri sonrası görüntünün üstüne yarı saydam boyar; arka plan olduğu gibi kalır."""
    karisim = (1 - alfa) * post.astype(np.float32) + alfa * PALET[L]
    return np.where((L > 0)[..., None], karisim, post).astype(np.uint8)

def gorsel_yaz(base, L, post):
    Image.fromarray(etiket_renklendir(L)).save(OUT / "label_renkli" / f"{base}.png")
    Image.fromarray(bindir(post, L)).save(OUT / "overlay" / f"{base}.jpg", quality=88)

def bos_oran(img, esik=15):
    """Öncesi/sonrası görüntüde veri olmayan (siyah) piksellerin oranı: RGB toplamı < esik."""
    return float((img.astype(np.int32).sum(-1) < esik).mean())

def manifest_oku_yukselt(man_path, fields):
    """Manifest'i okur ve güncel şemaya getirir; bitmiş (status=ok) karoların kümesini döndürür."""
    man_path = Path(man_path)
    if not man_path.exists():
        return set()
    rows = [r for r in csv.DictReader(open(man_path, encoding="utf8")) if r.get("status") == "ok"]
    for r in rows:
        if r.get("pre_blank_frac") in (None, ""):
            p = OUT / "pre" / f"{r['tile']}.jpg"
            r["pre_blank_frac"] = round(bos_oran(np.array(Image.open(p).convert("RGB"))), 4) if p.exists() else ""
    tmp = man_path.with_suffix(".csv.tmp")
    with open(tmp, "w", newline="", encoding="utf8") as f:
        wr = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore"); wr.writeheader(); wr.writerows(rows)
    tmp.replace(man_path)          # yarım yazılmış manifest bırakmamak için önce geçici dosyaya yazılır
    return {r["tile"] for r in rows}

## 3. BRIGHT etiketleri
`target.zip` (47 MB) tüm olayların etiketini içerir; yalnız `turkey-earthquake` karolarını alırız (1.114).

In [ ]:
tz_path = SCRATCH / "target.zip"
if not tz_path.exists():
    urllib.request.urlretrieve(HF + "target.zip", tz_path)
tz = zipfile.ZipFile(tz_path)
tz_lock = threading.Lock()
def tz_read(name):
    with tz_lock:
        return tz.read(name)
TILES = sorted(n for n in tz.namelist() if "turkey-earthquake" in n and n.endswith(".tif"))
print("Türkiye karo sayısı:", len(TILES))

## 4. Maxar olay-sonrası sahneler
Maxar STAC kataloğundan 6 Şubat 2023 sonrası edinimler ve içindeki öğeler (ilk çalıştırmada birkaç dakika sürebilir, sonra önbellekten gelir).

In [ ]:
col_url = MAXAR + "collection.json"
col = http_json(col_url)
ACQ = []
for l in col["links"]:
    if l["rel"] == "child":
        u = urljoin(col_url, l["href"]); c = http_json(u)
        dt = c["extent"]["temporal"]["interval"][0][0]
        if dt >= POST_FROM:
            ACQ.append((dt, u, c))
ACQ.sort(key=lambda x: x[0])

ITEMS = {}
def load_items(u, c):
    hs = [urljoin(u, l["href"]) for l in c["links"] if l["rel"] == "item"]
    with ThreadPoolExecutor(12) as ex:
        ITEMS[u] = list(ex.map(lambda h: dict(http_json(h), _url=h), hs))
for dt, u, c in ACQ:
    load_items(u, c)
print("olay-sonrası edinim:", len(ACQ), " toplam öğe:", sum(len(v) for v in ITEMS.values()))

## 5. Karo üretimi
Her karo için: BRIGHT öncesi optik + etiket okunur, karoyu tamamen kapsayan Maxar sahneleri arasından (tercihen ≤ 15 Şubat) **en küçük off-nadir**'li olanı seçilir, sonrası görüntü karo sınırlarına yeniden örneklenir.

In [ ]:
FIELDS = ["tile", "city", "split", "crs", "left", "bottom", "right", "top", "post_date", "off_nadir", "gsd",
          "post_asset", "blank_frac", "pre_blank_frac", "px_intact", "px_damaged", "px_destroyed", "status"]
man_path = OUT / "manifest.csv"
# Yalnızca status=ok olan karolar "bitmiş" sayılır; hatalı/kapsamsız olanlar yeniden denenir (2b'deki manifest_oku_yukselt).
DONE = manifest_oku_yukselt(man_path, FIELDS)
lock = threading.Lock(); tl = threading.local()

def tile_name(n):
    return os.path.basename(n).replace("_building_damage.tif", "")

def process(n, wr, t0, counter):
    if MAX_SECONDS and time.time() - t0 > MAX_SECONDS:
        return
    base = tile_name(n)
    if not hasattr(tl, "rz"):
        tl.rz = retry(lambda: RemoteZip(HF + "pre-event.zip"))
    rec = {k: "" for k in FIELDS}; rec["tile"] = base
    try:
        L = tifffile.imread(io.BytesIO(tz_read(n)))
        pre_bytes = retry(lambda: tl.rz.read(f"pre-event/{base}_pre_disaster.tif"))
        with MemoryFile(pre_bytes) as mf, mf.open() as ds:
            crs = ds.crs.to_string(); b = tuple(ds.bounds)
            pre = np.moveaxis(ds.read([1, 2, 3]), 0, -1)
        if L.shape != pre.shape[:2]:   # etiket ile öncesi görüntü aynı ızgarada olmalı
            raise ValueError(f"etiket/öncesi boyut uyuşmuyor: {L.shape} {pre.shape[:2]}")
        rec["pre_blank_frac"] = round(bos_oran(pre), 4)
        w, s, e, nn = transform_bounds(crs, "EPSG:4326", *b)
        city = min(CITIES, key=lambda k: dist_km(((w + e) / 2, (s + nn) / 2), CITIES[k]))
        cnt = np.bincount(L.ravel(), minlength=4)
        rec.update(city=city, split=SPLIT.get(city, "unused"), crs=crs, left=b[0], bottom=b[1], right=b[2], top=b[3],
                   px_intact=int(cnt[1]), px_damaged=int(cnt[2]), px_destroyed=int(cnt[3]))
        cands = []
        for dt, u, c in ACQ:
            if not any(x[0] <= w and x[2] >= e and x[1] <= s and x[3] >= nn for x in c["extent"]["spatial"]["bbox"]):
                continue
            for it in ITEMS[u]:
                q = it["bbox"]
                if q[0] <= w and q[2] >= e and q[1] <= s and q[3] >= nn:
                    cands.append((dt, it))
        if not cands:
            rec["status"] = "maxar_kapsam_yok"
        else:
            pref = [x for x in cands if x[0][:10] <= POST_PREFERRED_UNTIL] or cands
            dt, it = min(pref, key=lambda x: (x[1]["properties"].get("view:off_nadir") or 99))
            url = urljoin(it["_url"], it["assets"]["visual"]["href"])
            with rasterio.open(url) as src, WarpedVRT(src, crs=crs, resampling=Resampling.bilinear) as v:
                arr = v.read([1, 2, 3], window=from_bounds(*b, transform=v.transform), out_shape=(3, 1024, 1024),
                             resampling=Resampling.bilinear)
            post = np.moveaxis(arr, 0, -1).astype(np.uint8)
            rec.update(post_date=dt[:10], off_nadir=it["properties"].get("view:off_nadir"), gsd=it["properties"].get("gsd"),
                       post_asset=url, blank_frac=round(float((post.sum(-1) == 0).mean()), 4), status="ok")
            Image.fromarray(pre.astype(np.uint8)).save(OUT / "pre" / f"{base}.jpg", quality=92)
            Image.fromarray(post).save(OUT / "post" / f"{base}.jpg", quality=92)
            L8 = L.astype(np.uint8)
            if L8.max() > 3:
                raise ValueError(f"beklenmeyen sınıf indeksi: {np.unique(L8)}")
            Image.fromarray(L8).save(OUT / "label" / f"{base}.png")   # ham indeks: model/değerlendirme bunu okur
            gorsel_yaz(base, L8, post)                                  # insan için renkli kopya + bindirme
    except Exception as ex:
        rec["status"] = "hata:" + str(ex)[:80]
    with lock:
        wr.writerow(rec); counter[0] += 1
        if counter[0] % 50 == 0:
            print(counter[0], base, rec["status"], round(time.time() - t0), "s")

todo = [n for n in TILES if tile_name(n) not in DONE]
if LIMIT:
    todo = todo[:LIMIT]
print("işlenecek karo:", len(todo), "(zaten hazır:", len(DONE), ")")
new = not man_path.exists()
with open(man_path, "a", newline="", encoding="utf8") as f:
    wr = csv.DictWriter(f, fieldnames=FIELDS)
    if new:
        wr.writeheader()
    t0 = time.time(); counter = [0]
    with ThreadPoolExecutor(WORKERS) as ex:
        list(ex.map(lambda n: process(n, wr, t0, counter), todo))
print("bu çalıştırmada işlenen:", counter[0])

### 5b. Eksik renkli/bindirme görsellerini tamamla
Karo üretimi sırasında `label_renkli/` ve `overlay/` zaten yazılır. Bu hücre, bu özellikten önce üretilmiş karolar için (veya silinmiş dosyalar için) eksik olanları `label/` ve `post/`'tan yeniden üretir. Yeniden çalıştırmak güvenlidir: var olanlara dokunmaz.

In [ ]:
ok_tiles = [r["tile"] for r in csv.DictReader(open(man_path, encoding="utf8")) if r["status"] == "ok"]
eksik = [t for t in ok_tiles if not ((OUT / "label_renkli" / f"{t}.png").exists() and (OUT / "overlay" / f"{t}.jpg").exists())]
for t in eksik:
    L = np.array(Image.open(OUT / "label" / f"{t}.png"))
    post = np.array(Image.open(OUT / "post" / f"{t}.jpg").convert("RGB"))
    gorsel_yaz(t, L, post)
print(f"{len(ok_tiles)} karo, {len(eksik)} tanesi için görsel üretildi")

### 5c. Overlay'deki kaçırmalar: kayma ölçümü ve hizalama

**Sorun.** Bazı karolarda etiket konturları çatılara oturmuyor, kaymış görünüyor. İki neden var:
1. Etiketler öncesi (nadire yakın) görüntü geometrisine göre çizilmiş; bizim sonrası sahneler ise 6°–44° bakış açısıyla çekilmiş. Yüksek açıda çatı, taban izine göre *yükseklik × tan(açı)* kadar yer değiştirir (bina yatması) ve arazi modeli hatası ortorektifikasyona ek kayma ekler. Adıyaman ve Malatya (≈44°) en kötü etkilenenler.
2. Küçük, sistematik bir georeferans farkı: İslahiye 6° olduğu hâlde ≈5 m (15 piksel) tutarlı kayma var.

**Literatürde nasıl yapılıyor?** Ortak ilke: *referans = öncesi görüntü + etiket; hizalanan şey sonrası görüntüdür.*
- **xBD** (Gupta ve ark., 2019; arXiv:1911.09296, DOI 10.48550/arXiv.1911.09296): poligonlar öncesi görüntüde çizilir; sonrasıyla uyuşmayan olaylarda **olay başına sabit bir kayma sonrası görüntülere uygulanır**, poligonlar oynatılmaz. Off-nadir açı için ayrıca bir düzeltme yoktur, yalnız üstveridir.
- **BRIGHT** (Chen ve ark., Earth Syst. Sci. Data 17, 6217, 2025; DOI 10.5194/essd-17-6217-2025): sonrası görüntüyü (SAR) öncesi optik referansa QGIS Georeferencer ile, uzmanların elle seçtiği kontrol noktaları ve *Thin Plate Spline* dönüşümüyle hizalar, çapraz kontrol eder; vekil kayıt hatası ≈1 piksel (Türkiye 0,94). Etiketler öncesi geometride kalır. **Dikkat:** BRIGHT'ın Türkiye için sonrası *optik* görüntüsü yoktur (Tablo 8 yalnız beş olayı kapsar, Türkiye yok). Maxar sonrası görüntüsü bu çalışmaya özgüdür; bu yüzden BRIGHT yazarlarının onu nasıl hizaladığına dair bir referans yoktur.
- **Drone/CRASAR** (Manzini ve ark., 2024; arXiv:2405.06593, DOI 10.48550/arXiv.2405.06593): drone ortomozaiklerinde hata düzgün değildir (ortalama ≈82 piksel), tek tip doğrusal kayma yetmez, doğrusal olmayan yöntem gerekir. Uydu için tek tip kaymanın yaygın olduğunu belirtir. İkinci makalede (drone) önemli olacak.
- **Off-nadir bina çıkarımı** (Wang ve ark., 2022; arXiv:2204.13637, DOI 10.48550/arXiv.2204.13637): hizalamak yerine modele çatı maskesiyle birlikte çatıdan tabana kayma vektörünü de öğretir (BONAI).

**Bizim yaklaşım** xBD'ninkiyle aynı mantıkta: şehir başına sabit öteleme, referans öncesi/etiket, **sonrası görüntü taşınır**. Fark: xBD kaymayı koordinat sisteminden bilir, biz görüntü içeriğinden kestiriyoruz (kaynak açı kaynaklı). BRIGHT'ınki kadar hassas değildir: TPS ile ≈1 piksel hedeflenir, bizde şehir içi karo saçılımı 4–19 piksel.

**Neden etiketi değil sonrasını taşıyoruz?** İlk denememizde etiketi kaydırmıştık. Bu, etiketi sonrasıyla hizalar ama öncesiyle bozar; oysa pre+post kullanan kollar ve bina yerelleştirme öncesi görüntüye dayanır. Sonrasını etiketin/öncesinin çerçevesine taşıyınca üçlü (öncesi, sonrası, etiket) tek geometride kalır.

**Denediklerimiz.**
- *Daha düşük açılı sahne seçmek:* STAC taraması yapıldı, daha iyi sahne yok. Adıyaman/Malatya için aday sayısı 1–2 ve hepsi ≈44°.
- *Öncesi ile sonrası arasında faz korelasyonu:* güvenilmez (tepe zayıf, saçılım büyük). Kullanılmadı.
- *Etiket konturlarını sonrası görüntünün kenarlarına oturtmak:* tutarlı çıktı, kullanıldı.

**Yöntem.** Etiketin kontur piksellerinin altındaki görüntü gradyanı (kenar şiddeti) yüksek olmalıdır. `kontur_skoru` bunu ölçer: *kontur üstündeki ortalama gradyan ÷ görüntünün genel ortalama gradyanı* (1 = rastgele; yüksek = gerçek kenarlarda). `kayma_kestir`, FFT çapraz korelasyonuyla konturu ±64 piksel (≈21 m) aralığında kaydırıp skoru en çok artıran `(dx, dy)` değerini bulur. Karo bazlı kayma bina yüksekliğine göre dalgalandığı için her şehirde en çok 100 karonun **ortancası** alınır. Sınama: kayma karoların ilk yarısından kestirilir, ikinci yarıda skor karşılaştırılır. Skoru en az %5 iyileştirmeyen şehirde (ör. Antakya) kayma uygulanmaz.

**Dikkat edilecekler.**
- `label/` ve `pre/` **hiç değiştirilmedi**. Taşınmış sonrası görüntü `post_hizali/`'dir; kaydırmayla açığa çıkan kenar şeridi siyahtır (0). Değerlendirmede ve kayıpta bu şerit `gecerli_maske(dx, dy)` ile dışlanmalıdır.
- Kayma yalnız etiket geometrisi ve görüntü kenarlarından kestirildi; model çıktısı kullanılmadı, dolayısıyla test performansına sızıntı yoktur. Yine de test şehirlerinin kaymaları da veriden kestirilir; makalede ön işleme adımı olarak açıkça yazın.
- Tek öteleme bina yüksekliğine bağlı kalan hatayı gidermez (uzun bina daha çok kayar) ve karo içi doğrusal olmayan bozulmayı da. Bu yüzden bina düzeyi yerine tampon/karo düzeyi ölçüt önerisi geçerli kalıyor.
- İşaret kuralı: `hizalama.csv`'deki `dx > 0` etiket konturunun sonrası görüntüye oturması için sağa kayması gerektiğini söyler; bu yüzden sonrası görüntü `(-dx, -dy)` kadar taşınır.

In [ ]:
from PIL import ImageFilter
import pandas as pd

R_ARAMA = 64                                   # aranacak en büyük kayma (piksel) ≈ 21 m
(SCRATCH / "hizalama").mkdir(exist_ok=True)    # şehir başına sonuç önbelleği (yeniden çalıştırınca tekrar hesaplanmaz)

def kenar_haritasi(L):
    """Etiketli bölgenin sınır pikselleri (içi dolu pikselin 4 komşusundan biri boşsa)."""
    m = L > 0
    return m & ~(np.roll(m, 1, 0) & np.roll(m, -1, 0) & np.roll(m, 1, 1) & np.roll(m, -1, 1))

def gradyan(post):
    g = np.array(Image.fromarray(post).convert("L").filter(ImageFilter.GaussianBlur(1.2)), dtype=np.float32)
    gy, gx = np.gradient(g)
    return np.hypot(gx, gy)

def kontur_skoru(G, E, dx=0, dy=0):
    """Kontur üstündeki ortalama gradyan / görüntünün genel ortalaması (1 = rastgele; yüksek = kenara oturuyor)."""
    ys, xs = np.nonzero(E)
    return float(G[np.clip(ys + dy, 0, 1023), np.clip(xs + dx, 0, 1023)].mean() / G.mean())

def kayma_kestir(G, E):
    """Konturu (dx, dy) kaydırınca görüntü kenarlarına en iyi oturan değer (FFT çapraz korelasyonu)."""
    Eb = np.array(Image.fromarray((E * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.5)), dtype=np.float32)
    C = np.fft.ifft2(np.fft.fft2(G - G.mean()) * np.conj(np.fft.fft2(Eb - Eb.mean()))).real
    C = np.roll(np.roll(C, R_ARAMA, 0), R_ARAMA, 1)[:2 * R_ARAMA + 1, :2 * R_ARAMA + 1]   # sıfır kayma -> merkez
    i = np.unravel_index(C.argmax(), C.shape)
    return int(i[1] - R_ARAMA), int(i[0] - R_ARAMA)

def karo_yukle(t):
    post = np.array(Image.open(OUT / "post" / f"{t}.jpg").convert("RGB"))
    return gradyan(post), kenar_haritasi(np.array(Image.open(OUT / "label" / f"{t}.png")))

def sehir_kaymasi(city, ornek=100, tohum=0):
    p = SCRATCH / "hizalama" / f"{city.replace('/', '_')}.json"
    if p.exists():
        return json.loads(p.read_text())
    m = pd.read_csv(man_path)
    # yeterince etiketli ve sonrası görüntüsü dolu karolar (az etiketli karoda kayma kestirilemez)
    m = m[(m.city == city) & (m.status == "ok") & (m.blank_frac < 0.02) & (m.px_intact + m.px_damaged + m.px_destroyed > 8000)]
    tiles = list(m.sample(min(len(m), ornek), random_state=tohum).tile)
    d = np.array([kayma_kestir(*karo_yukle(t)) for t in tiles])
    h = len(tiles) // 2
    sdx, sdy = int(np.median(d[:h, 0])), int(np.median(d[:h, 1]))       # sınama: ilk yarıdan kestir, ikinci yarıda dene
    s0, s1 = [], []
    for t in tiles[h:h + 30]:
        G, E = karo_yukle(t); s0.append(kontur_skoru(G, E)); s1.append(kontur_skoru(G, E, sdx, sdy))
    px_m = float(((m.right - m.left) / 1024).median())
    r = dict(city=city, n_karo=len(tiles), dx=int(np.median(d[:, 0])), dy=int(np.median(d[:, 1])),
             dx_std=round(float(d[:, 0].std()), 1), dy_std=round(float(d[:, 1].std()), 1), piksel_m=round(px_m, 3),
             skor_kaymasiz=round(float(np.mean(s0)), 2), skor_sehir_kaymasi=round(float(np.mean(s1)), 2))
    r["kayma_m"] = round(float(np.hypot(r["dx"], r["dy"]) * px_m), 1)
    p.write_text(json.dumps(r))
    return r

In [ ]:
sehirler = sorted(pd.read_csv(man_path).city.unique())
hizalama = pd.DataFrame([sehir_kaymasi(c) for c in sehirler])
hizalama["uygula"] = hizalama.skor_sehir_kaymasi / hizalama.skor_kaymasiz >= 1.05   # %5'ten az iyileşiyorsa kaydırma
hizalama.to_csv(OUT / "hizalama.csv", index=False)
hizalama

In [ ]:
def kaydir(a, dx, dy, bos=0):
    """Dizi içeriğini dx piksel sağa, dy piksel aşağı kaydırır (H x W veya H x W x C); açığa çıkan şerit `bos` ile dolar."""
    out = np.full_like(a, bos)
    out[max(dy, 0):1024 + min(dy, 0), max(dx, 0):1024 + min(dx, 0)] = a[max(-dy, 0):1024 + min(-dy, 0), max(-dx, 0):1024 + min(-dx, 0)]
    return out

def gecerli_maske(dx, dy):
    """post_hizali içinde gerçek görüntü taşıyan (kaydırmayla dolmamış) piksellerin maskesi. Kayıp/metrikte bunun dışı dışlanır."""
    return kaydir(np.ones((1024, 1024), np.uint8), -dx, -dy).astype(bool)

for d in ("post_hizali", "overlay_hizali"):
    (OUT / d).mkdir(exist_ok=True)
kayma = {r.city: ((int(r.dx), int(r.dy)) if r.uygula else (0, 0)) for r in hizalama.itertuples()}
yeni = 0
for r in csv.DictReader(open(man_path, encoding="utf8")):
    t = r["tile"]
    if r["status"] != "ok" or ((OUT / "post_hizali" / f"{t}.jpg").exists() and (OUT / "overlay_hizali" / f"{t}.jpg").exists()):
        continue
    dx, dy = kayma[r["city"]]
    post = np.array(Image.open(OUT / "post" / f"{t}.jpg").convert("RGB")); L = np.array(Image.open(OUT / "label" / f"{t}.png"))
    ph = kaydir(post, -dx, -dy)                      # sonrası görüntüyü etiketin/öncesinin geometrisine taşı (label/ ve pre/ sabit)
    Image.fromarray(ph).save(OUT / "post_hizali" / f"{t}.jpg", quality=95)
    Image.fromarray(bindir(ph, L)).save(OUT / "overlay_hizali" / f"{t}.jpg", quality=88)
    yeni += 1
print("sonrası görüntüye uygulanan şehir kaymaları (dx, dy; sonrası -(dx, dy) taşınır):", kayma, "| üretilen karo:", yeni)

## 6. Kalite kontrol
Durum sayıları, bölme/şehir özeti, **bakış açısı** (off-nadir) ve **öncesi görüntüdeki boş (siyah) alan**. Şehir bölmesi aynı zamanda bakış açısı bölmesi olduğu için genelleme farkı iki etkiyi karıştırır.

In [ ]:
import pandas as pd
df = pd.read_csv(man_path)
print(df["status"].value_counts().to_string(), "\n")
df["yikik_var"] = df.px_destroyed > 0; df["hasarli_var"] = df.px_damaged > 0
ozet = (df[df.status == "ok"].groupby(["split", "city"])
        .agg(karo=("tile", "count"), yikik_karo=("yikik_var", "sum"), hasarli_karo=("hasarli_var", "sum"),
             off_nadir_medyan=("off_nadir", "median"), bos_5yuzde_ustu=("blank_frac", lambda s: int((s > 0.05).sum())),
             pre_bos_5yuzde_ustu=("pre_blank_frac", lambda s: int((s > 0.05).sum())),
             pre_bos_50yuzde_ustu=("pre_blank_frac", lambda s: int((s > 0.5).sum()))))
ozet

In [ ]:
import matplotlib.pyplot as plt
ok = df[df.status == "ok"]
ornek = []
for c in ["Antakya/Defne", "Adiyaman", "Islahiye"]:
    d = ok[ok.city == c].assign(h=lambda d: d.px_damaged + d.px_destroyed).sort_values("h")
    if len(d):
        ornek.append(d.tile.iloc[-min(5, len(d))])
if not ornek:  # küçük deneme (RAG_LIMIT) için
    ornek = list(ok.tile.iloc[:2])
fig, ax = plt.subplots(len(ornek), 3, figsize=(12, 4 * len(ornek)), squeeze=False)
for i, t in enumerate(ornek):
    A = np.array(Image.open(OUT / "pre" / f"{t}.jpg")); B = np.array(Image.open(OUT / "post" / f"{t}.jpg"))
    L = np.array(Image.open(OUT / "label" / f"{t}.png"))
    for j, (im, ad) in enumerate([(A, "öncesi"), (B, "sonrası"), (bindir(B, L), "etiket (yeşil sağlam, sarı hasarlı, kırmızı yıkık)")]):
        ax[i, j].imshow(im); ax[i, j].set_title(f"{t[-8:]} {ad}"); ax[i, j].axis("off")
plt.tight_layout(); plt.show()


### 6b. Etiket doğrulama
Üç şeyi doğrularız: (1) her `label/` dosyasında yalnız 0–3 değerleri var, (2) `manifest.csv` içindeki sınıf piksel sayıları PNG'lerle birebir tutuyor, (3) aynı etiket üç şekilde çizildiğinde ne gördüğümüz: ham indeks (siyah görünen), kontrastı açılmış (×85) ve renkli.

In [ ]:
degerler, uyumsuz = set(), []
for r in csv.DictReader(open(man_path, encoding="utf8")):
    if r["status"] != "ok":
        continue
    L = np.array(Image.open(OUT / "label" / f"{r['tile']}.png"))
    degerler |= set(np.unique(L).tolist())
    c = np.bincount(L.ravel(), minlength=4)
    if (c[1], c[2], c[3]) != (int(r["px_intact"]), int(r["px_damaged"]), int(r["px_destroyed"])):
        uyumsuz.append(r["tile"])
print("etiket değerleri:", sorted(degerler), "(yalnız 0-3 olmalı)")
print("manifest ile PNG sayımı uyuşmayan karo:", len(uyumsuz))
assert degerler <= {0, 1, 2, 3} and not uyumsuz

t = df[df.status == "ok"].assign(h=lambda d: d.px_damaged + d.px_destroyed).sort_values("h").tile.iloc[-1]
L = np.array(Image.open(OUT / "label" / f"{t}.png"))
print(t, {SINIF_ADLARI[k]: int(v) for k, v in enumerate(np.bincount(L.ravel(), minlength=4))})
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (im, ad, kw) in zip(ax, [(L, "ham indeks (0-255 ölçeğinde siyah)", dict(cmap="gray", vmin=0, vmax=255)),
                                (L * 85, "×85 ile kontrast açılmış", dict(cmap="gray", vmin=0, vmax=255)),
                                (etiket_renklendir(L), "palet ile renkli", {})]):
    a.imshow(im, **kw); a.set_title(ad); a.axis("off")
plt.tight_layout(); plt.show()

### 6c. Hizalama doğrulama
(1) `post_hizali/`'nin geçerli bölgesi, orijinal sonrası görüntünün tam olarak `(-dx, -dy)` kaydırılmış hâliyle uyuşuyor (fark yalnız JPEG yeniden sıkıştırma düzeyinde); (2) kaydırmayla açığa çıkan şerit siyah; (3) `label/` ve `pre/` dokunulmamış (etiket değerleri hâlâ 0–3). (4) En çok kaymış şehirden bir karoda, **aynı etiket konturunun** orijinal sonrası görüntüde (beyaz) ve hizalı sonrası görüntüde (camgöbeği) görünümü.

In [ ]:
hata = []
for r in csv.DictReader(open(man_path, encoding="utf8")):
    if r["status"] != "ok":
        continue
    t = r["tile"]; dx, dy = kayma[r["city"]]; m = gecerli_maske(dx, dy)
    po = np.array(Image.open(OUT / "post" / f"{t}.jpg").convert("RGB")).astype(int)
    ph = np.array(Image.open(OUT / "post_hizali" / f"{t}.jpg").convert("RGB")).astype(int)
    ref = kaydir(po, -dx, -dy)
    fark = np.abs(ph - ref)[m].mean()
    serit = ph[~m].mean() if (~m).any() else 0.0
    L = np.array(Image.open(OUT / "label" / f"{t}.png"))
    if fark > 4 or serit > 6 or L.max() > 3:
        hata.append((t, round(float(fark), 2), round(float(serit), 2)))
print("hizalama doğrulama: sorunlu karo =", len(hata), hata[:3]); assert not hata

en = hizalama.assign(m=np.hypot(hizalama.dx, hizalama.dy)).sort_values("m").city.iloc[-1]
t = df[(df.city == en) & (df.status == "ok")].assign(n=lambda d: d.px_intact + d.px_damaged + d.px_destroyed).sort_values("n").tile.iloc[-5]
L = np.array(Image.open(OUT / "label" / f"{t}.png"))
po = np.array(Image.open(OUT / "post" / f"{t}.jpg").convert("RGB")); ph = np.array(Image.open(OUT / "post_hizali" / f"{t}.jpg").convert("RGB"))
def kontur(img, renk):
    o = img.copy(); o[kenar_haritasi(L)] = renk; return o
sl = (slice(300, 700), slice(300, 700))        # ayrıntı için orta 400x400 kesit
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(kontur(po, (255, 255, 255))[sl]); ax[0].set_title(f"{en}: orijinal sonrası + etiket konturu (beyaz)")
ax[1].imshow(kontur(ph, (0, 255, 255))[sl]); ax[1].set_title(f"{en}: hizalı sonrası + aynı kontur (camgöbeği)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 7. Sonraki adımlar için notlar
- **Etiket ve öncesi:** her zaman `label/` (ham indeks, 0–3) ve `pre/`; ikisi hiç değişmez. `label_renkli/`, `overlay/` ve `overlay_hizali/` yalnızca inceleme içindir.
- **Sonrası görüntü:** modeller/VLM için `post_hizali/` kullanılması önerilir (etiket ve öncesiyle aynı geometri); kaydırmayla açığa çıkan siyah şerit `gecerli_maske(dx, dy)` ile kayıp ve metrikten dışlanır. Şehir kaymaları `hizalama.csv`'de. Makalede hizalamalı ve hizalamasız (`post/`) sonucu birlikte vermek en savunulabilir yoldur.
- **Değerlendirme ölçütü:** bina düzeyi yerine tampon/karo düzeyi; tek öteleme bina yüksekliğine bağlı kalan hatayı gidermez.
- **Öncesi görüntü boşluğu:** `pre_blank_frac` yüksek karolarda (ör. > 0,5) öncesi görüntü büyük ölçüde boştur. Öncesi/sonrası karşılaştıran kollar bu karolarda adaletsiz davranır. Karolar **silinmedi** (bölme sayıları değişmesin diye); karar (dışlama, maske ya da yalnız sonrası kullanan kollarda tutma) 02. adımda verilecek ve makalede raporlanacaktır.
- **Çözünürlük:** karo kenarı ≈312–358 m, 1024 piksele örneklendiği için piksel ≈0,3–0,35 m; Maxar GSD ≈0,6–0,9 m olduğundan sonrası görüntü yaklaşık 2× yukarı örneklenmiştir (gerçek ayrıntı sanıldığından azdır).